# Task 1 — Data Preparation & Hospital Simulation
## Federated Learning for Heart Disease (Option 4)

This notebook covers:
1. Loading and exploring the Heart Disease dataset
2. Cleaning, encoding, and normalizing features
3. Simulating 8 federated hospitals (IID and non-IID)
4. Visualizing data distributions across hospitals

---

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Image

# Make src/ importable
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

print('Environment ready.')

## 1. Dataset Exploration

In [ ]:
df = pd.read_csv('data/heart.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print(f'\nMissing values:\n{df.isnull().sum()}')

In [ ]:
print('Descriptive statistics:')
df.describe()

In [ ]:
# Feature descriptions
feature_descriptions = {
    'age'     : 'Age (years)',
    'sex'     : 'Sex (1=Male, 0=Female)',
    'cp'      : 'Chest pain type (0-3)',
    'trestbps': 'Resting blood pressure (mm Hg)',
    'chol'    : 'Serum cholesterol (mg/dl)',
    'fbs'     : 'Fasting blood sugar > 120 mg/dl (1=True)',
    'restecg' : 'Resting ECG results (0-2)',
    'thalach' : 'Max heart rate achieved',
    'exang'   : 'Exercise induced angina (1=Yes)',
    'oldpeak' : 'ST depression induced by exercise',
    'slope'   : 'Slope of peak exercise ST segment (0-2)',
    'ca'      : 'Number of major vessels (0-4)',
    'thal'    : 'Thalassemia type (0-3)',
    'target'  : 'Heart disease diagnosis (1=Sick, 0=Healthy)',
}
pd.DataFrame.from_dict(feature_descriptions, orient='index', columns=['Description'])

## 2. Data Preparation (Cleaning, Encoding, Normalization)

In [ ]:
from data_preparation import prepare_dataset

X, y, feature_names, scaler = prepare_dataset(save=True)

print(f'\nFinal shapes:')
print(f'  X: {X.shape}  (samples x features after one-hot encoding)')
print(f'  y: {y.shape}')

In [ ]:
# Show the processed feature matrix
X_df = pd.DataFrame(X, columns=feature_names)
print(f'Processed features ({len(feature_names)} columns):')
X_df.head()

## 3. Federated Partitioning — 8 Simulated Hospitals

In [ ]:
from federated_partition import partition_data, N_CLIENTS

splits = partition_data(n_clients=N_CLIENTS, alpha=0.5, seed=42)

In [ ]:
# Summary table
rows = []
for mode, split_list in splits.items():
    for i, (Xi, yi) in enumerate(split_list):
        n_sick = int(yi.sum())
        rows.append({
            'Mode'     : mode.upper(),
            'Hospital' : f'H{i+1}',
            'Samples'  : len(yi),
            'Sick (1)' : n_sick,
            'Healthy (0)': len(yi) - n_sick,
            '% Sick'   : f'{n_sick/len(yi)*100:.1f}%',
        })

summary_df = pd.DataFrame(rows)
summary_df

## 4. Distribution Visualizations

In [ ]:
from visualize_distribution import generate_all_figures
fig_paths = generate_all_figures()

In [ ]:
# Figure 1: Global class distribution
display(Image(fig_paths[0]))

In [ ]:
# Figure 2: Samples per hospital (IID vs non-IID)
display(Image(fig_paths[1]))

In [ ]:
# Figure 3: IID per-client class distribution
display(Image(fig_paths[2]))

In [ ]:
# Figure 4: non-IID per-client class distribution
display(Image(fig_paths[3]))

## 5. Summary

| Step | Status |
|------|--------|
| Dataset loaded | 1025 samples, 13 features, 0 missing |
| Categorical encoding | One-hot: cp, restecg, slope, thal (13 -> 23 features) |
| Normalization | StandardScaler on 5 continuous features |
| IID partition | 8 hospitals, ~128 samples each, ~50% sick |
| non-IID partition | 8 hospitals, Dirichlet(alpha=0.5), strong imbalance |
| Figures | 4 PNG files saved to outputs/figures/ |

### Data Files Produced
```
outputs/
  processed/
    X.npy                (1025, 23)   preprocessed features
    y.npy                (1025,)      binary labels
    feature_names.txt                 23 feature names
  clients_iid/
    client_0_X.npy ... client_7_X.npy
    client_0_y.npy ... client_7_y.npy
  clients_noniid/
    client_0_X.npy ... client_7_X.npy
    client_0_y.npy ... client_7_y.npy
  figures/
    global_class_distribution.png
    samples_per_client.png
    class_dist_iid.png
    class_dist_noniid.png
```

> These files are ready to be consumed by the other team members (FedAvg, FedProx, FedAdam, SGD implementation tasks).